# Drowsiness Detection Using Facial Landmarks

**Objective:** To detect prolonged eye closure using facial landmarks and identify possible signs of drowsiness in real time.


In [1]:
import cv2
import numpy as np
import mediapipe as mp
import time

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = "face_landmarker.task"

base_options = python.BaseOptions(
    model_asset_path=model_path
)

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1
)

landmarker = vision.FaceLandmarker.create_from_options(options)

print("Face Landmarker loaded successfully!")

Face Landmarker loaded successfully!


In [3]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Camera could not be opened.")
else:
    print("Camera opened successfully!")

cap.release()

Camera opened successfully!


In [4]:
# Eye landmark indices
LEFT_EYE_TOP = 159
LEFT_EYE_BOTTOM = 145

RIGHT_EYE_TOP = 386
RIGHT_EYE_BOTTOM = 374

print("Eye landmarks defined successfully!")

Eye landmarks defined successfully!


In [5]:
def eye_opening(landmarks, top_index, bottom_index, width, height):
    top = landmarks[top_index]
    bottom = landmarks[bottom_index]

    top_point = np.array([top.x * width, top.y * height])
    bottom_point = np.array([bottom.x * width, bottom.y * height])

    distance = np.linalg.norm(top_point - bottom_point)

    return distance

print("Eye measurement function created successfully!")

Eye measurement function created successfully!


In [6]:
EYE_CLOSED_THRESHOLD = 8
CLOSED_FRAMES_REQUIRED = 15

print("Drowsiness threshold set successfully!")
print("Eye threshold:", EYE_CLOSED_THRESHOLD)
print("Required closed frames:", CLOSED_FRAMES_REQUIRED)

Drowsiness threshold set successfully!
Eye threshold: 8
Required closed frames: 15


In [7]:
cap = cv2.VideoCapture(0)

closed_frames = 0

print("Starting Drowsiness Detection...")
print("Press 'q' to quit.")

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        print("Failed to read camera frame.")
        break

    frame = cv2.flip(frame, 1)

    height, width, _ = frame.shape

    # Convert OpenCV image to MediaPipe image
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    # Detect facial landmarks
    result = landmarker.detect(mp_image)

    status = "AWAKE"

    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        # Calculate eye opening
        left_eye = eye_opening(
            landmarks,
            LEFT_EYE_TOP,
            LEFT_EYE_BOTTOM,
            width,
            height
        )

        right_eye = eye_opening(
            landmarks,
            RIGHT_EYE_TOP,
            RIGHT_EYE_BOTTOM,
            width,
            height
        )

        average_eye = (left_eye + right_eye) / 2

        # Check whether eyes are closed
        if average_eye < EYE_CLOSED_THRESHOLD:
            closed_frames += 1
        else:
            closed_frames = 0

        # Drowsiness decision
        if closed_frames >= CLOSED_FRAMES_REQUIRED:
            status = "DROWSY - EYES CLOSED"

        # Display eye measurement
        cv2.putText(
            frame,
            f"Eye Opening: {average_eye:.1f}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    cv2.putText(
        frame,
        f"Status: {status}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255) if "DROWSY" in status else (0, 255, 0),
        2
    )

    cv2.imshow("Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

print("Drowsiness detection stopped.")

Starting Drowsiness Detection...
Press 'q' to quit.
Drowsiness detection stopped.


In [8]:
drowsiness_events = 0

print("Drowsiness event counter initialized!")

Drowsiness event counter initialized!


In [9]:
print("=" * 45)
print("       DROWSINESS DETECTION REPORT")
print("=" * 45)

print(f"Detection method      : Eye landmark analysis")
print(f"Eye threshold         : {EYE_CLOSED_THRESHOLD}")
print(f"Required closed frames: {CLOSED_FRAMES_REQUIRED}")
print(f"Drowsiness events     : {drowsiness_events}")

if drowsiness_events == 0:
    print("Result                : No prolonged eye closure detected")
else:
    print("Result                : Possible drowsiness detected")

print("=" * 45)

       DROWSINESS DETECTION REPORT
Detection method      : Eye landmark analysis
Eye threshold         : 8
Required closed frames: 15
Drowsiness events     : 0
Result                : No prolonged eye closure detected


In [10]:
cap = cv2.VideoCapture(0)

closed_frames = 0
drowsiness_events = 0
drowsy_already_counted = False

print("Starting Drowsiness Detection...")
print("Press 'q' to quit.")

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        print("Failed to read camera frame.")
        break

    frame = cv2.flip(frame, 1)

    height, width, _ = frame.shape

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    result = landmarker.detect(mp_image)

    status = "AWAKE"

    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        left_eye = eye_opening(
            landmarks,
            LEFT_EYE_TOP,
            LEFT_EYE_BOTTOM,
            width,
            height
        )

        right_eye = eye_opening(
            landmarks,
            RIGHT_EYE_TOP,
            RIGHT_EYE_BOTTOM,
            width,
            height
        )

        average_eye = (left_eye + right_eye) / 2

        if average_eye < EYE_CLOSED_THRESHOLD:
            closed_frames += 1
        else:
            closed_frames = 0
            drowsy_already_counted = False

        if closed_frames >= CLOSED_FRAMES_REQUIRED:
            status = "DROWSY - EYES CLOSED"

            if not drowsy_already_counted:
                drowsiness_events += 1
                drowsy_already_counted = True

        cv2.putText(
            frame,
            f"Eye Opening: {average_eye:.1f}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    cv2.putText(
        frame,
        f"Status: {status}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255) if "DROWSY" in status else (0, 255, 0),
        2
    )

    cv2.putText(
        frame,
        f"Events: {drowsiness_events}",
        (20, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 0),
        2
    )

    cv2.imshow("Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

print("Drowsiness detection stopped.")
print("Total drowsiness events:", drowsiness_events)

Starting Drowsiness Detection...
Press 'q' to quit.
Drowsiness detection stopped.
Total drowsiness events: 7


In [11]:
print("=" * 50)
print("          DROWSINESS DETECTION REPORT")
print("=" * 50)

print(f"Detection method       : Facial landmark analysis")
print(f"Eye closure threshold  : {EYE_CLOSED_THRESHOLD}")
print(f"Required closed frames : {CLOSED_FRAMES_REQUIRED}")
print(f"Drowsiness events      : {drowsiness_events}")

if drowsiness_events > 0:
    print("Result                 : Possible drowsiness detected")
else:
    print("Result                 : No prolonged eye closure detected")

print("=" * 50)

          DROWSINESS DETECTION REPORT
Detection method       : Facial landmark analysis
Eye closure threshold  : 8
Required closed frames : 15
Drowsiness events      : 7
Result                 : Possible drowsiness detected


## Conclusion

In this practical, MediaPipe Face Landmarker and OpenCV were used to develop a real-time drowsiness detection system. Facial landmarks were used to measure eye opening, and prolonged eye closure was identified as a possible sign of drowsiness.

During testing, the system successfully detected prolonged eye-closure events and displayed the drowsiness status on the webcam feed.

This project demonstrates how facial landmark analysis can be applied to real-time driver and fatigue monitoring applications.